# Create Fritz Thyssen Stiftung Awards (GRANT PATTERN, method-5 static HTML)

Fritz Thyssen Stiftung "funding" records from the foundation's own WordPress archive at fritz-thyssen-stiftung.de. Major German humanities/social-sciences funder (Geisteswissenschaften und Sozialwissenschaften), funding research projects since 1959.

**Prerequisites:**
- Run `scripts/local/fritz_thyssen_to_s3.py` first.

**Data source:** Discovery method-5 static HTML via WordPress YOAST sub-sitemaps. The sitemap index lists 4 sub-sitemaps but in practice only `wp-sitemap-posts-funding-1.xml` returns 200 (the others are 404, likely a stale W3 Total Cache reference). The reachable sitemap has **2,000 funding URLs**.

Each funding page (~99KB) has clean labeled fields:
- `<h1>` project title (in German)
- `<div class="info-box fundingbox">` — Institution: PI name + affiliation
- `<p class="funding-detail">Bewilligung | YEAR</p>` — approval year
- `<p class="funding-detail">Förderbereich | TOPIC</p>` — funding area

**S3 location:** `s3a://openalex-ingest/awards/fritz_thyssen/fritz_thyssen_fundings.parquet`

**Awarding body:**
- funder_id: 4320321876
- display_name: "Fritz Thyssen Stiftung"
- country: DE
- ROR: none
- DOI: 10.13039/501100003390

**Coverage (smoke test 10 fundings, 2026-05-23):**
- 100% title / slug
- 90% pi_raw / bewilligung_year / foerderbereich
- 60% institution (some pages omit the affiliation line)

The script's `funder_award_id = ft-{slug}` is guaranteed unique by WordPress slug constraint.

**Amount/currency:** NULL with §6.7 waiver. Fritz Thyssen publishes program-level fixed amounts on /foerderung/ pages (research grants typically EUR 100k-500k) but does NOT publish per-funding amount on the archive. Grant/research-funder precedent: HHMI #44, CIFAR #79, Damon Runyon #73, Packard #95, Rita Allen #107, Schmidt Sciences #108, NOMIS #109, Wenner-Gren #110, Mercator #116.

**Provenance:** `fritz_thyssen_fundings`.

**Priority:** 117 (UCOP 106 - Mercator 116 are immediately-prior slots in flight; Vilcek at 105 is the current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fritz_thyssen_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fritz_thyssen/fritz_thyssen_fundings.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.fritz_thyssen_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.fritz_thyssen_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Fritz Thyssen is non-monetary in our schema (§6.7 waiver). The runbook §1.5 RLIKE money scan is run for completeness.

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.fritz_thyssen_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.fritz_thyssen_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT title, slug, pi_raw, institution, bewilligung_year, foerderbereich
FROM openalex.awards.fritz_thyssen_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.fritz_thyssen_raw;


In [ ]:
%sql
-- Year distribution. Expected: long tail going back to 1959-ish.
SELECT bewilligung_year, COUNT(*) as fundings
FROM openalex.awards.fritz_thyssen_raw
WHERE bewilligung_year IS NOT NULL
GROUP BY bewilligung_year ORDER BY bewilligung_year DESC LIMIT 30;


In [ ]:
%sql
-- Top funding areas
SELECT foerderbereich, COUNT(*) as cnt
FROM openalex.awards.fritz_thyssen_raw
GROUP BY foerderbereich ORDER BY cnt DESC LIMIT 20;


## Step 1.6: Fail-fast — Verify the Fritz Thyssen Funder Row Exists

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320321876;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fritz_thyssen_awards
USING delta
AS
WITH
ft_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321876
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(bewilligung_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(bewilligung_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.fritz_thyssen_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        COALESCE(NULLIF(TRIM(g.foerderbereich), ''), 'Fritz Thyssen Funding') as funder_scheme,

        'fritz_thyssen_fundings' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- lead_investigator: PI name + institution affiliation
        CASE
            WHEN g.pi_given_name IS NULL AND g.pi_family_name IS NULL AND g.institution IS NULL THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(g.pi_given_name), '') as given_name,
                NULLIF(TRIM(g.pi_family_name), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(g.institution), '') as name,
                    'DE' as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN ft_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 117

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fritz_thyssen_fundings' AND priority = 117;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    117 as priority
FROM openalex.awards.fritz_thyssen_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.fritz_thyssen_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.fritz_thyssen_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(landing_page_url) as has_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) as pct_pi
FROM openalex.awards.fritz_thyssen_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       lead_investigator.affiliation.name AS pi_affiliation,
       landing_page_url
FROM openalex.awards.fritz_thyssen_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.fritz_thyssen_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.fritz_thyssen_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;


In [ ]:
%sql
-- §6.7 amount/currency check (waivered)
SELECT COUNT(*) AS total, COUNT(amount) AS has_amount,
       ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
       COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.fritz_thyssen_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.fritz_thyssen_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids,
       COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.fritz_thyssen_awards;
